# PPO for Continuous Control: Humanoid (MuJoCo)

CSCI 6353 · Topic 41.

The **same continuous PPO** as Topic 40 (Gaussian policy, clipped objective, GAE, K-epoch
reuse), scaled up to a 3D **MuJoCo humanoid**: a 348-number state and 17 joint torques. The
only real additions are **observation normalization** (essential at this scale) and a bigger
256x256 network.

Humanoid is a **hard** task: a good walking policy needs several million steps. On CPU this is
slow; the run below is shortened so you can watch it start to stand and move.

## 1. Setup  (Humanoid needs MuJoCo)

In [ ]:
!pip -q install "gymnasium[mujoco]" torch matplotlib

In [ ]:
import gymnasium as gym
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.distributions import Normal

LR, GAMMA, LMBDA, EPS_CLIP = 3e-4, 0.99, 0.95, 0.2
ROLLOUT, K_EPOCH, MINIBATCH = 4096, 10, 128
S_DIM, A_DIM = 348, 17

## 2. Observation normalization + the Gaussian actor-critic

`RMS` keeps a running mean/std of the 348 observations and normalizes them (Humanoid barely
learns without this). The network is the Topic-40 actor-critic widened to 256 units, with a
17-dimensional Gaussian policy.

In [ ]:
class RMS:
    def __init__(self, d):
        self.mean = np.zeros(d); self.var = np.ones(d); self.count = 1e-4
    def update(self, x):
        bm, bv, bc = x.mean(0), x.var(0), x.shape[0]
        d = bm - self.mean; tot = self.count + bc
        self.mean += d * bc / tot
        self.var = (self.var*self.count + bv*bc + d**2*self.count*bc/tot) / tot
        self.count = tot
    def norm(self, x):
        return np.clip((x - self.mean) / np.sqrt(self.var + 1e-8), -10, 10)

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(S_DIM, 256); self.fc2 = nn.Linear(256, 256)
        self.fc_mu = nn.Linear(256, A_DIM)
        self.log_std = nn.Parameter(-0.5 * torch.ones(A_DIM))
        self.v1 = nn.Linear(S_DIM, 256); self.v2 = nn.Linear(256, 256); self.v3 = nn.Linear(256, 1)
    def pi(self, x):
        h = torch.tanh(self.fc1(x)); h = torch.tanh(self.fc2(h))
        return self.fc_mu(h), torch.exp(self.log_std).expand_as(self.fc_mu(h))
    def v(self, x):
        h = torch.tanh(self.v1(x)); h = torch.tanh(self.v2(h))
        return self.v3(h).squeeze(-1)

## 3. Train  (step-based PPO with done-masked GAE — identical to Topic 40)

Raise `TOTAL_STEPS` toward 4,000,000+ for a real walk; the default is a short taste.

In [ ]:
TOTAL_STEPS = 400_000    # a taste; use 4_000_000+ to see it walk
env = gym.make('Humanoid-v5')
net = ActorCritic(); opt = optim.Adam(net.parameters(), lr=LR); rms = RMS(S_DIM)
s, _ = env.reset(seed=0)
ep_ret, recent, curve, gstep = 0.0, [], [], 0
while gstep < TOTAL_STEPS:
    Sn, A, R, LP, D, V = [], [], [], [], [], []
    for _ in range(ROLLOUT):
        rms.update(s[None]); sn = rms.norm(s).astype(np.float32)
        with torch.no_grad():
            mu, std = net.pi(torch.from_numpy(sn)); dist = Normal(mu, std)
            a = dist.sample(); logp = dist.log_prob(a).sum().item(); v = net.v(torch.from_numpy(sn)).item()
        s2, r, term, trunc, _ = env.step(np.clip(a.numpy(), -0.4, 0.4))
        done = term or trunc
        Sn.append(sn); A.append(a.numpy()); R.append(r); LP.append(logp); D.append(float(done)); V.append(v)
        s = s2; ep_ret += r; gstep += 1
        if done:
            recent.append(ep_ret); recent = recent[-50:]; ep_ret = 0.0; s, _ = env.reset()
    with torch.no_grad():
        last_v = net.v(torch.from_numpy(rms.norm(s).astype(np.float32))).item()
    Sn = torch.tensor(np.array(Sn)); A = torch.tensor(np.array(A), dtype=torch.float); LP = torch.tensor(LP, dtype=torch.float)
    R = np.array(R, np.float32); D = np.array(D, np.float32); V = np.array(V, np.float32)
    adv = np.zeros(ROLLOUT, np.float32); gae = 0.0
    for t in reversed(range(ROLLOUT)):
        nv = last_v if t == ROLLOUT-1 else V[t+1]; nt = 1.0 - D[t]
        delta = R[t] + GAMMA*nv*nt - V[t]; gae = delta + GAMMA*LMBDA*nt*gae; adv[t] = gae
    ret = torch.tensor(adv + V); adv = torch.tensor((adv - adv.mean())/(adv.std()+1e-8))
    idx = np.arange(ROLLOUT)
    for _ in range(K_EPOCH):
        np.random.shuffle(idx)
        for i in range(0, ROLLOUT, MINIBATCH):
            mb = idx[i:i+MINIBATCH]
            mu, std = net.pi(Sn[mb]); dist = Normal(mu, std)
            ratio = torch.exp(dist.log_prob(A[mb]).sum(1) - LP[mb])
            s1 = ratio*adv[mb]; s2c = torch.clamp(ratio, 1-EPS_CLIP, 1+EPS_CLIP)*adv[mb]
            loss = -torch.min(s1, s2c).mean() + 0.5*F.mse_loss(net.v(Sn[mb]), ret[mb])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 0.5); opt.step()
    mr = float(np.mean(recent)) if recent else float('nan')
    curve.append((gstep, mr)); print(f'step {gstep:8d}  mean50 {mr:8.1f}')
env.close()

## 4. The learning curve

In [ ]:
import matplotlib.pyplot as plt
xs, ys = zip(*curve)
plt.figure(figsize=(8,4.5))
plt.plot(xs, ys, color='#7c3aed', lw=2.2)
plt.xlabel('environment steps'); plt.ylabel('mean return (last 50 episodes)')
plt.title('Continuous PPO on Humanoid-v5'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Where to go next

- **Watch it.** Make the env with `render_mode='rgb_array'`, roll out the trained `net` greedily
  (use `mu`), and save the frames as a GIF to see the gait.
- **Assignment 41 — HalfCheetah.** Change the env to `HalfCheetah-v5` (17-dim state, 6-dim action,
  no falling), keep observation normalization, and train it to run. Submit the learning curve and
  a render. With a Gaussian policy, a new continuous-control task is mostly a new environment string.